# Data Understanding & Quality Audit

## Objective

理解实验复盘所依赖的四张核心表，确认数据粒度、关键字段、连接关系与质量检查范围。本 Notebook 记录数据审计口径；具体 SQL 位于 `../sql/01_data_quality.sql`，不在此修改原始数据。

## Core Tables

| Table | Grain | Primary key | Key fields and purpose |
|---|---|---|---|
| `website_sessions` | 一行一个网站 Session | `website_session_id` | `created_at`、`user_id`、`is_repeat_session`、`utm_source`、`utm_campaign`、`utm_content`、`device_type`、`http_referer`；描述访问者、设备和流量属性 |
| `website_pageviews` | 一行一个页面浏览 | `website_pageview_id` | `created_at`、`website_session_id`、`pageview_url`；恢复 Session 内页面顺序并识别 Billing 曝光 |
| `orders` | 一行一个订单 | `order_id` | `created_at`、`website_session_id`、`user_id`、`primary_product_id`、`items_purchased`、`price_usd`、`cogs_usd`；识别购买并计算收入 |
| `order_item_refunds` | 一行一条退款记录 | `order_item_refund_id` | `created_at`、`order_item_id`、`order_id`、`refund_amount_usd`；计算退款金额和净收入 |

完整字段字典位于 `../data/reference/maven_fuzzy_factory_data_dictionary.csv`。

## Table Relationships

```text
website_sessions (1)
    ├──< website_pageviews (many)
    └── orders (0 or 1 in the imported schema)
            └──< order_item_refunds (many)
```

- `website_pageviews.website_session_id` → `website_sessions.website_session_id`
- `orders.website_session_id` → `website_sessions.website_session_id`
- `order_item_refunds.order_id` → `orders.order_id`

实验样本从 `website_pageviews` 的 Billing 曝光开始，再连接 Session 属性、曝光后订单和订单退款。

## Data Quality Checks

`01_data_quality.sql` 依次检查：

1. 四张表行数与 `created_at` 时间范围。
2. 各表主键重复和主键缺失。
3. 核心分析字段的 NULL 情况。
4. Pageview → Session、Order → Session、Refund → Order 的孤儿记录。
5. `device_type`、`is_repeat_session`、`utm_source`、`utm_campaign` 和 `pageview_url` 的取值分布。
6. `items_purchased`、`price_usd`、`cogs_usd` 和 `refund_amount_usd` 的基础合理性。

UTM 字段允许因自然或直接流量出现 NULL，因此不应不加判断地当作数据错误。审计过程只报告问题，不覆盖或清洗原始 CSV。

## Audit Boundary

- 原始数据保存在本地 `data/raw/`，不会提交到 GitHub。
- 数据导入脚本使用可替换的本地路径占位符。
- 数据质量检查先于实验样本构建。
- 发现的数据限制需要保留到实验有效性和业务结论中，而不是通过任意删除记录来隐藏。